In [1]:
import pathlib
from typing import Sequence, Callable, List

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import equinox as eqx
from jaxtyping import Array, Float, Int, PyTree, PRNGKeyArray
import numpyro.distributions as dist

import torch
from torch.utils.data import DataLoader, Subset

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.util

# import ipdb

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# input_directory = pathlib.Path.cwd().parent / "light curves" / "Simulations"
# input_directory = pathlib.Path.cwd().parent / "light curves"
# input_directory = pathlib.Path.cwd()
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
len(ds)

12983

In [6]:
rng_key = jr.PRNGKey(42)

parameters = {
    'training': {
        'splitting': {
            'test_size':  0.2,
            'random_state': 42
        },
        'batch_size': 32,
        'learning_rate': 1e-3,
        # 'learning_rate': 3e-5,
        # 'learning_rate': 1e-6,
    },
}

In [7]:
train_idx, validation_idx = train_test_split(
    np.arange(len(ds)), 
    **parameters['training']['splitting'], 
    shuffle=True,
    stratify=ds.labels
)
train_ds, validation_ds = Subset(ds, train_idx), Subset(ds, validation_idx)

# a random key...
key, subkey = jr.split(rng_key)

# ...to compute a seed for a PyTorch `Generator`...
torch_seed = jr.bits(subkey, (1,), dtype=jnp.uint32).item()
torch_rng = torch.Generator().manual_seed(torch_seed)

# ...meant for "batching" in the `DataLoader`'s
train_dl = DataLoader(train_ds, parameters['training']['batch_size'], shuffle=True, generator=torch_rng)
validation_dl = DataLoader(validation_ds, batch_size=parameters['training']['batch_size'], shuffle=False, generator=torch_rng)

In [8]:
batch = next(iter(train_dl))

In [9]:
signals_len = ds[0][1].shape[-1]
signals_len

10000

In [10]:
tuple(ds[0][1].shape)

(1, 10000)

In [11]:
parameters['encoder'] = {
    'input_size': ds[0][1].shape,
}

In [12]:
parameters['model'] = {
    # 'dim_z': 4,
    'dim_z': 10,
    # 'dim_z': 20,
    'kernel_sizes': [128, 64, 32],
    'channels': [12, 12, 12],
    'stride': 2,
}

In [13]:
class Encoder(eqx.Module):

    layers: list
    layers_spatial_sizes: List[int]

    def __init__(self, input_size: Sequence[int], dim_z: int, kernel_sizes: List[int], channels: List[int], stride: int, key: PRNGKeyArray):

        # for the sake of readability
        n_conv_layers = len(channels)
        
        assert len(kernel_sizes) == n_conv_layers, f'number of kernel sizes ({len(kernel_sizes)}) must match number of channels ({n_conv_layers})'

        padding = 0

        # for the sake of readability
        n_input_maps, input_maps_len = input_size

        # the number of input channels is given by the data
        channels = [n_input_maps] + channels

        self.layers = []

        conv_output_len = input_maps_len

        self.layers_spatial_sizes = [input_maps_len]

        for in_channels, out_channels, key, kernel_size in zip(
            channels[:-1], channels[1:],
            jr.split(key, n_conv_layers),
            kernel_sizes
        ):
            
            # the length of the feature map for the next layer
            conv_output_len = trapallada.util.conv_output_size(conv_output_len, (kernel_size,), (stride,), (padding,))
            print(f'{conv_output_len=}')

            # key, subkey = jr.split(key)
            
            self.layers.extend([
                eqx.nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, key=key),
                jax.nn.relu,
                eqx.nn.LayerNorm((out_channels, conv_output_len[0]))
            ])

            self.layers_spatial_sizes.append(conv_output_len[0])

        
        self.layers_spatial_sizes.append(out_channels * conv_output_len[0])

        # the last PRNG key from the loop is split again
        _, subkey = jr.split(key)

        self.layers.extend([
            jnp.ravel,
            eqx.nn.Linear(in_features=self.layers_spatial_sizes[-1], out_features=2*dim_z, key=subkey),
            lambda x: x.at[dim_z:].set(jax.nn.softplus(x[dim_z:]) + 1e-3)
            # lambda x: x.at[dim_z:].set(jax.nn.softplus(x[dim_z:]))
        ])
        
    def __call__(self, x: Float[Array, '1 length']) -> Float[Array, 'twice_dim_z']:

        for layer in self.layers:

            x = layer(x)

        return x

    def debug_call(self, x: Float[Array, '1 length']) -> Float[Array, 'twice_dim_z']:

        for layer in self.layers:

            x = layer(x)

            print(f'after {layer.__module__}: {x.shape}')

In [14]:
x = batch[1][0].numpy()
encoder = Encoder(**(parameters['model'] | parameters['encoder']), key=jr.PRNGKey(42))
encoder.debug_call(x)
print(encoder)

conv_output_len=(4937,)
conv_output_len=(2437,)
conv_output_len=(1203,)
after equinox.nn._conv: (12, 4937)
after jax._src.nn.functions: (12, 4937)
after equinox.nn._normalisation: (12, 4937)
after equinox.nn._conv: (12, 2437)
after jax._src.nn.functions: (12, 2437)
after equinox.nn._normalisation: (12, 2437)
after equinox.nn._conv: (12, 1203)
after jax._src.nn.functions: (12, 1203)
after equinox.nn._normalisation: (12, 1203)
after jax._src.numpy.lax_numpy: (14436,)
after equinox.nn._linear: (20,)
after __main__: (20,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,1,128],
      bias=f32[12,1],
      in_channels=1,
      out_channels=12,
      kernel_size=(128,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(12, 4937),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[12,4937],
      bias=f32[12,4937]
    ),
   

In [15]:
encoder.layers_spatial_sizes

[10000, 4937, 2437, 1203, 14436]

In [16]:
z = encoder(x)
z.shape

(20,)

In [17]:
parameters['decoder'] = {'layers_spatial_sizes': encoder.layers_spatial_sizes}

In [18]:
class Decoder(eqx.Module):

    layers: list

    def __init__(self, dim_z: int, layers_spatial_sizes: List[int], kernel_sizes: List[int], channels: List[int], stride: int, key: PRNGKeyArray):

        # for the sake of readability
        n_conv_layers = len(channels)

        assert len(kernel_sizes) == n_conv_layers, f'number of kernel sizes ({len(kernel_sizes)}) must match number of channels ({n_conv_layers})'
        assert (len(kernel_sizes) == len(layers_spatial_sizes) - 2,
                f'number of kernel sizes and channels ({len(kernel_sizes)}) must match that of spatial sizes ({len(layers_spatial_sizes)}) minus 2')

        padding = 0

        # the final number of output channels must be 1
        channels = list(reversed(channels)) + [1]

        # the architecture of the encoder is reversed
        kernel_sizes = list(reversed(kernel_sizes))
        layers_spatial_sizes = list(reversed(layers_spatial_sizes))

        self.layers = []

        key, subkey = jr.split(key)

        self.layers.extend([
            eqx.nn.Linear(in_features=dim_z, out_features=layers_spatial_sizes[0], key=subkey),
            jax.nn.relu,
            lambda x: jnp.reshape(x, (channels[0], layers_spatial_sizes[1])), # from a 1D vector to a 2D array
        ])

        # after the `reshape`ing
        conv_output_len = layers_spatial_sizes[1]

        for in_channels, out_channels, key, kernel_size, expected_len in zip(
            channels[:-1], channels[1:],
            jr.split(key, n_conv_layers),
            kernel_sizes,
            layers_spatial_sizes[2:],
        ):
            

            conv_output_len = trapallada.util.transp_conv_output_size(conv_output_len, (kernel_size,), (stride,), (padding,))
            # print(f'{conv_output_len=}, {expected_len=}')

            output_padding = expected_len - conv_output_len[0]
            
            key, subkey = jr.split(key)
            
            self.layers.extend([
                eqx.nn.ConvTranspose1d(
                    in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride,
                    output_padding=output_padding, key=subkey),
                jax.nn.relu,
                eqx.nn.LayerNorm((out_channels, expected_len))
            ])

            conv_output_len += output_padding

        # last `relu` and `LayerNorm` are not needed
        del self.layers[-2:]
        
    def __call__(self, x: Float[Array, 'dim_z']) -> Float[Array, '1 length']:

        for layer in self.layers:

            x = layer(x)

        return x

    def debug_call(self, x: Float[Array, 'dim_z']) -> Float[Array, '1 length']:

        for layer in self.layers:

            x = layer(x)

            print(f'after {layer.__module__}: {x.shape}')

<>:11: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:11: SyntaxWarning: assertion is always true, perhaps remove parentheses?
/tmp/ipykernel_110961/55100250.py:11: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert (len(kernel_sizes) == len(layers_spatial_sizes) - 2,


In [19]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
decoder = Decoder(**(parameters['model'] | parameters['decoder']), key=jr.PRNGKey(42))
decoder.debug_call(z_mean)

after equinox.nn._linear: (14436,)
after jax._src.nn.functions: (14436,)
after __main__: (12, 1203)
after equinox.nn._conv: (12, 2437)
after jax._src.nn.functions: (12, 2437)
after equinox.nn._normalisation: (12, 2437)
after equinox.nn._conv: (12, 4937)
after jax._src.nn.functions: (12, 4937)
after equinox.nn._normalisation: (12, 4937)
after equinox.nn._conv: (1, 10000)


In [20]:
class VAE(eqx.Module):

    encoder: Encoder
    decoder: Decoder

    def __init__(self, encoder: Encoder, decoder: Decoder):

        self.encoder = encoder
        self.decoder = decoder

    def __call__(self, x: Float[Array, '1 length'], key: PRNGKeyArray) -> PyTree:

        z_mean, z_sd = jnp.split(self.encoder(x), 2, axis=-1)
        # z_sd = jnp.exp(z_log_var / 2)

        # z = z_mean + z_sd * jr.normal(jr.PRNGKey(42), mu.shape)

        z = dist.Normal(loc=z_mean, scale=z_sd).sample(key)

        return self.decoder(z), z_mean, z_sd


In [21]:
vae = VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((1, 10000), (10,), (10,))

In [22]:
x_pred

Array([[-0.05094664,  0.01958639, -0.04254634, ...,  0.00485079,
        -0.00609281, -0.02103021]], dtype=float32)

In [23]:
# jax.vmap(vae)(batch[1].numpy(), jr.split(jr.PRNGKey(42), len(batch[1])))

In [24]:
def loss(
        model: eqx.Module,
        x: Float[Array, 'batch 1 length'],
        x_sd: float, key: PRNGKeyArray
        ) -> tuple[Float[Array, ''], tuple[Float[Array, ''], Float[Array, '']]]:

    # # for the sake of readability
    batch_size = len(x)

    x_pred, z_mean, z_sd = jax.vmap(model)(x, jr.split(key, batch_size))

    # jax.debug.print(x_pred)
    # print(f'{x_pred=}')

    # reconstruction term
    loglikelihood = dist.Normal(loc=x_pred, scale=x_sd).log_prob(x).sum() / batch_size

    # KL divergence
    kl = -0.5 * jnp.sum(1. + 2.*jnp.log(z_sd) - z_mean**2 - z_sd**2) / batch_size

    # return -loglikelihood, (loglikelihood, kl)
    return -loglikelihood + kl, (loglikelihood, kl)

In [25]:
loss(vae, batch[1].numpy(), 0.3, jr.PRNGKey(42))

(Array(60884.645, dtype=float32),
 (Array(-60881., dtype=float32), Array(3.6455855, dtype=float32)))

In [26]:
@eqx.filter_jit
def take_step(
    model: eqx.Module,
    opt_state: PyTree,
    x: Float[Array, 'batch 1 length'],
    x_sd: float,
    key: PRNGKeyArray
) -> tuple[tuple[eqx.Module], PyTree, Float[Array, '']]:

    value_aux, grads = eqx.filter_value_and_grad(loss, has_aux=True)(model, x, x_sd, key)

    # jax.debug.print(f'{grads=}')

    # it returns transformed/processed gradients, and the optimizer state
    updates, opt_state = optim.update(grads, opt_state, model)

    model = eqx.apply_updates(model, updates)

    return model, opt_state, value_aux

In [27]:
# parameters['loss'] = {'x_sd': 0.3}
parameters['loss'] = {'x_sd': 0.5}

In [28]:
# take_step(vae, opt_state, batch[1].numpy(), parameters['loss']['x_sd'], jr.PRNGKey(42))

In [29]:
# parameters['training']['learning rate'] = 1e-3
parameters['training']['learning rate'] = 3e-5
# parameters['training']['learning rate'] = 1e-6

In [30]:
encoder = Encoder(**(parameters['model'] | parameters['encoder']), key=jr.PRNGKey(42))
decoder = Decoder(**(parameters['model'] | parameters['decoder']), key=jr.PRNGKey(43))
model = VAE(encoder, decoder)

optim = optax.adamw(parameters['training']['learning_rate'])
opt_state = optim.init(eqx.filter(model, eqx.is_array))

conv_output_len=(4937,)
conv_output_len=(2437,)
conv_output_len=(1203,)


In [31]:
# model.encoder.layers[0].weight

In [32]:
# def check_for_nans(tree, name):
#     def check_leaf(leaf):
#         if jnp.isnan(leaf).any():
#             print(f"NaNs detected in {name}")
#     jax.tree_util.tree_map(check_leaf, tree)
def check_for_nans(tree, name):
    def check_leaf(path, leaf):
        if jnp.isnan(leaf).any():
            print(f"NaNs detected in {path}")
    jax.tree_util.tree_map_with_path(check_leaf, tree)

In [33]:
parameters['training']['n_epochs'] = 500
# parameters['training']['n_epochs'] = 100
# parameters['training']['n_epochs'] = 20
# parameters['training']['n_epochs'] = 6
# parameters['training']['n_epochs'] = 1

# results_df = trapallada.util.make_results_df(parameters['training']['n_epochs'])
results_df = trapallada.util.make_results_df(parameters['training']['n_epochs'], metrics=['loss', 'loglikelihood', 'kl'])

key=jr.PRNGKey(14)

for epoch in range(parameters['training']['n_epochs']):

    for i_batch, (_, x, _) in enumerate(train_dl):

        # print(model.encoder.layers[0].weight[0, 0, 0])

        key, subkey = jr.split(key)

        x = x.numpy()

        model, opt_state, (loss_value, (log_likelihood_value, kl_value)) = take_step(model, opt_state, x, parameters['loss']['x_sd'], subkey)

        check_for_nans(loss_value, "loss_value")
        check_for_nans(log_likelihood_value, "log_likelihood_value")
        check_for_nans(kl_value, "kl_value")
        check_for_nans(eqx.filter(model, eqx.is_array), "model parameters")

        # print((loss_value, (log_likelihood_value, kl_value)))

        # adding up *averages* (of both the loss and the accuracy)
        results_df.loc[epoch, ('training', 'loss')] += loss_value.item()
        results_df.loc[epoch, ('training', 'loglikelihood')] += log_likelihood_value.item()
        results_df.loc[epoch, ('training', 'kl')] += kl_value.item()

        # print(jax.vmap(model)(x, jr.split(jr.PRNGKey(42), len(x))))


    results_df.loc[
        epoch, [('training', 'loss'), ('training', 'loglikelihood'), ('training', 'kl')]
        ] /= len(train_dl)

    # validation
    # results_df.loc[epoch, [('validation', 'loss'), ('validation', 'accuracy')]] = evaluate(model, validation_dl)

    print(results_df.loc[epoch, 'training'].to_frame().T)

    # print(
    #     epoch,
    #     results_df.loc[epoch, ('training', 'loss')],
    #     # results_df.loc[epoch, ('validation', 'loss')],
    #     '|||',
    #     # results_df.loc[epoch, ('training', 'accuracy')],
    #     # results_df.loc[epoch, ('validation', 'accuracy')],
    # )

           loss  loglikelihood         kl
0  15392.131878  -15141.603699  250.52817
          loss  loglikelihood          kl
1  13433.29591  -13196.060216  237.235677
           loss  loglikelihood         kl
2  12879.715457  -12645.944537  233.77092
           loss  loglikelihood          kl
3  12472.637263   -12250.66692  221.970349
           loss  loglikelihood         kl
4  12195.051385  -11979.447037  215.60433
           loss  loglikelihood          kl
5  11962.950382  -11750.700856  212.249506
           loss  loglikelihood          kl
6  11747.021508  -11540.710412  206.311102
           loss  loglikelihood          kl
7  11582.360129  -11376.392533  205.967587
           loss  loglikelihood          kl
8  11401.579633  -11203.652806  197.926843
           loss  loglikelihood          kl
9  11289.023248  -11089.650844  199.372416
            loss  loglikelihood          kl
10  11128.014126   -10934.42695  193.587178
           loss  loglikelihood          kl
11  10993.52174  

KeyboardInterrupt: 

In [43]:
i = 10
x_pred, z_mean, z_sd = model(x[i], jr.PRNGKey(42))

df = pd.DataFrame({'prediction': x_pred[0], 'actual': x[i].ravel()})
n_rows = 2000
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

alt.Chart(...)

In [ ]:
z_mean, z_sd

(Array([-4.876185  , -0.9243267 ,  0.07822041,  7.818194  ,  6.159013  ,
        -0.09283876, -2.4463122 ,  3.1144423 ,  0.33242446,  5.8871527 ],      dtype=float32),
 Array([0.001     , 0.001     , 0.001     , 0.001     , 0.001     ,
        0.001     , 0.001     , 0.001     , 0.001     , 0.00100001],      dtype=float32))

In [ ]:
model.encoder(x[0])

Array([-4.8761849e+00, -9.2432672e-01,  7.8220412e-02,  7.8181939e+00,
        6.1590128e+00, -9.2838764e-02, -2.4463122e+00,  3.1144423e+00,
        3.3242446e-01,  5.8871527e+00,  1.0000000e-03,  1.0000000e-03,
        1.0000000e-03,  1.0000000e-03,  1.0000000e-03,  1.0000000e-03,
        1.0000000e-03,  1.0000000e-03,  1.0000000e-03,  1.0000068e-03],      dtype=float32)

*Untrained* encoder

In [ ]:
encoder(x[0])

Array([-0.20006207,  0.02075418,  0.14633936,  0.17731577, -0.57935756,
        0.6286559 ,  0.5903792 , -0.28655717, -0.1291584 , -0.30230722,
        0.6376333 ,  0.6382993 ,  1.2838753 ,  1.2649897 ,  0.3215992 ,
        0.6959488 ,  0.78274304,  0.86585635,  0.75800765,  0.90086013],      dtype=float32)